# Role Restoration: package-only pipeline

This notebook runs the local `role_restoration` package without importing scripts from `experiments/`. It supports model selection and executes the geometry and intervention stages for GPT-OSS.

The current package uses GPT-OSS Harmony channel tokens (`<|start|>`, `<|message|>`, `<|end|>`) and its attack parser expects Harmony tool calls. Qwen3 can be selected and loaded, but the notebook stops before analysis until a Qwen role-format adapter is added.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'role_restoration':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / 'notebook_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Outputs:', OUTPUT_DIR)

## Choose the model and experiment size

In [ ]:
# Change this value to load the other model.
MODEL_NAME = 'gpt-oss-20b'

MODEL_IDS = {
    'gpt-oss-20b': 'openai/gpt-oss-20b',
    'qwen3-8b': 'Qwen/Qwen3-8B',
}

CACHE_DIR = None  # Or a local Hugging Face cache path.
SEED = 123
LAYERS = (4, 8, 12, 16, 20)
PROJECTION_LAYER = 12
ROLE_TEXTS = 24
N_CARRIERS = 3
N_BENIGN_REFERENCE = 8
N_PAIRS = 24
N_ATTACK_SAMPLES = 1

if MODEL_NAME not in MODEL_IDS:
    raise ValueError(f'MODEL_NAME must be one of {sorted(MODEL_IDS)}')
MODEL_ID = MODEL_IDS[MODEL_NAME]
print('Selected:', MODEL_NAME, MODEL_ID)

In [ ]:
from role_restoration import base, geometry, intervention, plots

model, tok = base.load_model_and_tokenizer(
    model_id=MODEL_ID,
    cache_dir=CACHE_DIR,
    attn_implementation='eager',
)
print('Loaded:', MODEL_ID)

In [ ]:
def require_harmony_compatibility(tokenizer):
    required = ['<|start|>', '<|message|>', '<|end|>']
    missing = [token for token in required if tokenizer.convert_tokens_to_ids(token) is None]
    if missing:
        raise RuntimeError(
            f'{MODEL_NAME} loaded, but this package cannot analyze it yet. '
            f'role_restoration.base.render_single_role, spans, and parse_harmony use GPT-OSS '
            f'Harmony tokens; missing {missing}. Add a model adapter before running geometry '
            f'or interventions.'
        )
    if MODEL_NAME != 'gpt-oss-20b':
        raise RuntimeError(
            f'{MODEL_NAME} loaded, but the current package workflow is validated only for '
            'gpt-oss-20b Harmony formatting. Qwen requires a role renderer, span finder, '
            'and tool-call parser adapter.'
        )
    print('Package compatibility check passed.')

require_harmony_compatibility(tok)

## Stage 1: geometry

This computes role means, the role basis, `v_tag`, `v_style`, reliability controls, cosine/overlap metrics, and token projections.

In [ ]:
geometry_result = geometry.run_geometry(
    model=model,
    tok=tok,
    outdir=str(OUTPUT_DIR / 'geometry'),
    seed=SEED,
    layers=LAYERS,
    n_pairs=N_PAIRS,
    proj_layer=PROJECTION_LAYER,
)

geometry_result['geom'].round(3)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

projection_data = geometry_result['token_projections'].copy()
projection_data['condition'] = projection_data.apply(
    lambda row: f"{'CoT-style' if row['style'] == 'styled' else 'Plain'} text tagged {row['tag'].upper()}",
    axis=1,
)
condition_order = [
    'CoT-style text tagged USER',
    'Plain text tagged USER',
    'CoT-style text tagged COT',
    'Plain text tagged COT',
]
condition_means = (
    projection_data.groupby('condition', as_index=False)[['p_tag', 'p_style']]
    .mean()
    .set_index('condition')
    .reindex(condition_order)
)

print(f"Geometry layer: {PROJECTION_LAYER}")
print(f"cos(v_tag, v_style) = {float(geometry_result['geom'].loc[geometry_result['geom']['layer'] == PROJECTION_LAYER, 'cos(tag, style)'].iloc[0]):.4f}")
display(condition_means.round(3))

fig, ax = plt.subplots(figsize=(8, 6))
colors = {
    'CoT-style text tagged USER': '#2c3e50',
    'Plain text tagged USER': '#2c3e50',
    'CoT-style text tagged COT': '#c0392b',
    'Plain text tagged COT': '#c0392b',
}
markers = {
    'CoT-style text tagged USER': 'o',
    'Plain text tagged USER': 'x',
    'CoT-style text tagged COT': 'o',
    'Plain text tagged COT': 'x',
}
for condition, row in condition_means.iterrows():
    ax.scatter(
        row['p_tag'], row['p_style'], s=140,
        color=colors[condition], marker=markers[condition],
        edgecolors='black' if markers[condition] == 'o' else None,
        linewidths=0.8, label=condition,
    )
    ax.annotate(condition, (row['p_tag'], row['p_style']), xytext=(7, 7),
                textcoords='offset points', fontsize=9)
ax.axhline(0, color='#bbbbbb', linewidth=0.8)
ax.axvline(0, color='#bbbbbb', linewidth=0.8)
ax.set_xlabel('Projection onto tag direction (v_tag)')
ax.set_ylabel('Projection onto style direction (v_style)')
ax.set_title(f'Tag/style geometry at layer {PROJECTION_LAYER}')
ax.legend(fontsize=8, loc='best')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'geometry' / f'four_condition_projections_L{PROJECTION_LAYER}.png', dpi=200)
plt.show()


## Stage 2: role restoration and controls

In [ ]:
carriers, benign_pages, _ = base.load_wikipedia_material(
    seed=SEED,
    n_carriers=N_CARRIERS,
    n_benign_ref=N_BENIGN_REFERENCE,
    n_role_texts=ROLE_TEXTS,
)

mu_tool = base.benign_tool_reference(model, tok, benign_pages, LAYERS)
v_style = {layer: geometry_result['styd'][layer]['v_style'] for layer in LAYERS}
dirs = intervention.build_dirs(
    role_means=geometry_result['means'],
    mu_ref=mu_tool,
    layers=LAYERS,
    seed=SEED,
    v_style=v_style,
)

for layer in LAYERS:
    print(f'L{layer}: style inside role subspace = {dirs[layer]["style_inside_frac"]:.3f}')

In [ ]:
# Load the complete attack set from the original paper sources.
# A large n_* requests every available template because base.load_published_attacks
# internally uses min(requested_count, number_of_templates).
attacks = base.load_published_attacks(
    carriers=carriers,
    seed=SEED,
    n_prefix_user=10_000,
    n_prefix_other=10_000,
)
print('Total attacks:', len(attacks))
print(attacks.groupby(['family', 'type']).size().to_string())

specs = [
    intervention.Spec('none', (), 'none'),
    intervention.Spec('restore_role', LAYERS, 'restore'),
    intervention.Spec('restore_random', LAYERS, 'restore_rand'),
    intervention.Spec('ablate_style', LAYERS, 'ablate', direction='style'),
    intervention.Spec('ablate_style_on', LAYERS, 'ablate', direction='style_on'),
    intervention.Spec('ablate_style_perp', LAYERS, 'ablate', direction='style_perp'),
]

store = intervention.Store(OUTPUT_DIR / 'intervention_results.jsonl')
intervention.run_grid(
    model=model,
    tok=tok,
    attacks=attacks,
    specs=specs,
    dirs=dirs,
    n_samples=N_ATTACK_SAMPLES,
    experiment='notebook_intervention_all_attacks',
    store=store,
    seed=SEED,
    max_steps=3,
    max_new=400,
    temperature=1.0,
)

results = store.frame()
results.head()

In [ ]:
summary = intervention.summarize(results, 'notebook_intervention_all_attacks')
paired = intervention.paired_report(results, 'notebook_intervention_all_attacks', seed=SEED)
display(summary)
display(paired)

## Optional package-only diagnostics

These cells measure whether the edit moves injected-token role coordinates toward the benign tool reference and test benign utility.

In [ ]:
mediation_result = intervention.mediation(
    model=model,
    tok=tok,
    attacks=attacks.head(8),
    specs=[specs[0], specs[1], specs[2]],
    dirs=dirs,
    measure_layers=LAYERS,
)
display(mediation_result.groupby(['cond', 'layer']).dist_from_tool_ref.mean().round(3))

In [ ]:
plots.figure2_geometry(
    geometry_result['geom'],
    str(OUTPUT_DIR / 'figures'),
    tokens=geometry_result['token_projections'],
    proj_layer=PROJECTION_LAYER,
)
print('Figure written under', OUTPUT_DIR / 'figures')